# End-to-end Machine Learning Report

**Team 23:** Veronica Joe (jj3470), Crystal Guo (lg3434), Shuxuan Xu (sx2412), Fangyi Lin (fl2748)

**Github Repository:** https://github.com/xsxsx-999/5243-project4


## 1. Project Overview

### 1.1 Introduction

The objective of this project is to build an end-to-end machine learning pipeline to decode loan decisions within the United States mortgage market. Using data from the Home Mortgage Disclosure Act (HMDA), we aim to perform a comprehensive analysis of the factors driving application outcomes. By engineering robust features and training predictive models, this study seeks to identify the key drivers—ranging from applicant demographics to loan characteristics—that influence whether a mortgage application is ultimately approved or denied. 

### 1.2 Data Collection

To construct a focused and computationally viable dataset, our data acquisition and sampling strategy followed a strict, predefined methodology:

* **Data Source & Geographical Scope:** We obtained historical mortgage application records from the HMDA database spanning a six-year period from **2019 to 2024**. To capture representative trends while managing data volume, the geographical scope was restricted to two of the largest and most dynamic housing markets in the U.S.: **California (CA) and Texas (TX)**.

* **Target Formulation & Filtering:** The full HMDA dataset contains multiple stages of the application lifecycle (e.g., withdrawn, incomplete). For the purpose of binary classification, we isolated records to strictly two definitive outcomes:
    * **Originated/Approved** (`action_taken = 1`), mapped to `target_y = 1`
    * **Denied** (`action_taken = 3`), mapped to `target_y = 0`

* **Stratified Sampling:** Given the massive scale of the raw dataset, we applied a **1% stratified proportional sampling fraction** (`SAMPLE_FRACTION = 0.01`) strictly within the restricted two-class subset, which guarantees that the *class proportion* of the Originated vs. Denied subset is perfectly preserved. This yielded a final analytical sample of **139,991** records (`HMDA_CA_TX_2019_2024_Master.csv`). 

### 1.3 Statistical Interpretation & Caveats:
The models developed in this study are explicitly trained to estimate the conditional probability of origination: 

$$P(\text{Originated} \mid \text{Originated or Denied})$$

They do *not* estimate the unconditional probability $P(\text{Originated})$ over all applications. All downstream interpretations in this report—including model performance metrics, feature importance rankings, and fairness analyses—are strictly conditional on the population having already reached a definitive "Originated or Denied" resolution.

## 2. Cleaning & Preprocessing the data

To ensure data integrity and prevent data leakage, our preprocessing pipeline computes all **data-driven** transformations strictly on the **training set** (e.g., bin edges, imputations, log-transform decisions, and outlier boundaries), then applies the learned parameters to the test set. We also cache expensive intermediate artifacts (post-imputation) to keep the notebook reproducible and fast to re-run.

### 2.1 Datatype Formatting & Leakage Removal

- **Leakage Removal**: We dropped post-outcome and resolution-only fields so the model only uses information available at application time. This includes `action_taken` (basis of the target), `denial_reason-1..4` (only populated on denials), and `purchaser_type` (only recorded after origination).

- **Type Casting / Robust Missing Handling**: We explicitly cast known continuous fields to numeric and cast categorical fields to string while preserving missingness.

- **Underwriting DTI Feature Engineering (pre-drop, pre-split)**: We convert HMDA’s categorical DTI strings into two modeling features:
  - `dti_missing_flag`: binary indicator for missing / coded-missing values (e.g., `Exempt`, `<NA>`)
  - `debt_to_income_ratio_ord`: ordinal numeric mapping for the DTI buckets

  The raw `debt_to_income_ratio` string column is then dropped to avoid redundant high-cardinality behavior downstream.

### 2.2 Drop Columns
To reduce noise, prevent the curse of dimensionality, and optimize computational efficiency, we applied a strict, multi-tier pruning strategy. Features were systematically dropped based on missingness, variance, and cardinality thresholds:

#### 2.2.1 High Missingness
Any column with a missing value rate exceeding **80%** was directly dropped, as imputing such heavily sparse features would introduce severe artificial bias.

#### 2.2.2 Near-Zero Variance
Features lacking sufficient variation provide no predictive signal. For continuous columns, we dropped those with a standard deviation $\le 0.03$. For categorical columns, features were dropped if a single dominant category accounted for $\ge 98\%$ of the observations.

#### 2.2.3 High Cardinality & Tiered Filtering

To prevent sparse matrix explosion during One-Hot Encoding, we evaluated the unique value counts of categorical features. For the remaining categorical candidates, we applied a tiered threshold logic:

##### a. Auto-Drop ($>8$ unique values)
    
Highly fragmented categorical columns that exceeded 8 unique values were automatically dropped to minimize noise and multicollinearity.

##### b. Review Window ($5-8$ unique values)
    
Features containing between 5 and 8 unique values were flagged and isolated. Rather than dropping them outright, they were passed to the next pipeline stage for manual Exploratory Data Analysis (EDA). Rare or logically similar categories within these features are bucketed to improve model robustness. 

#### Details discussed as follow. !!!!!may move to appendix?

![image-2.png](attachment:image-2.png)

Based on the visual EDA of approval rates across the isolated categorical features, we executed the following consolidation strategy to maximize predictive signal while minimizing dimensionality *(Detailed distribution charts are available in the Appendix)*:

* **Retained as-is:** `covid_phase` (distinct macroeconomic trends) and `derived_ethnicity` (retained specifically to facilitate downstream Fair Lending analysis).
* **Binarized by Presence/Type:** * `co-applicant_sex`: Gender showed zero variance in approval outcomes. The true signal was simply the presence of a co-applicant. Converted to a binary `is_joint_application` (0 = Single, 1 = Joint).
  * `derived_loan_product_type`: Grouped into `First_Lien` (>80% approval) vs. `Subordinate_Lien` (~50% approval).
* **Binned by Risk/Approval Rates:**
  * `aus-1`: Grouped official systems into `Standard_AUS` (codes 1-4, ~90% approval) and mapped the rest to `Non_Standard_or_Exempt` (~60% approval).
  * `loan_purpose`: Grouped standard applications (purchase, refi) into `Standard_Purpose`, and higher-denial applications (home improvement, other) into `High_Risk_Purpose`.
  * `manufactured_home_land_property_interest`: Consolidated into `Has_Property_Interest` and `Not_Applicable_or_Exempt` based on distinct denial rate clusters.
* **Demographic Bucketing:** `applicant_age` was collapsed into broader life-stages (`<25`, `25-44`, `>44`). Anomaly placeholder values (8888, 9999) were explicitly replaced with `NaN` for safe downstream imputation.
* **Dropped:** `applicant_sex` (perfectly flat approval rates across categories; zero predictive value) and `applicant_ethnicity-1` (100% redundant with `derived_ethnicity`).

### 2.3 Feature Consolidation

To improve model interpretability and the model building process, we converted `activity_year` into a broader macroeconomic feature, `covid_phase` (<=2019: Pre-Pandemic; 2020–2021: Peak-Pandemic; >=2022: Post-Pandemic). Also, guided by EDA on approval rates, we logically grouped several sparse categorical features to reduce dimensionality and amplify predictive signal:

* `aus-1` $\rightarrow$ `aus_grouped` (Standard_AUS vs. Non_Standard_or_Exempt)
* `applicant_age` $\rightarrow$ `applicant_age_grouped` (Mapped undefined codes 8888/9999 to `NaN`)
* `derived_loan_product_type` $\rightarrow$ `loan_product_grouped` (First_Lien, Subordinate_Lien, Other)
* `manufactured_home_land_property_interest` $\rightarrow$ `manufactured_home_grouped`
* `co-applicant_sex` $\rightarrow$ `is_joint_application` (Binary)
* `loan_purpose` $\rightarrow$ `loan_purpose_grouped` (High_Risk vs. Standard)

### 2.4 Imputation
Missing values were visualized using a heatmap on a training sample to understand missingness patterns.

![image.png](attachment:image.png)

Our final imputation strategy is:

1. **Numeric imputation (KNN, train-only fit)**: We apply **KNNImputer (k=5)** to numeric columns (fit on training data only). To keep runtime manageable, KNN is only applied to numeric columns that contain missing values plus a small set of strongly related numeric predictors.

2. **Categorical imputation (mode, train-only fit)**: Categorical columns are imputed using the most frequent category. For scikit-learn compatibility, pandas string columns with `<NA>` are converted to `object` and missing values are represented as `np.nan` before imputation.

3. **Forced median override for discrete underwriting variables**: Two variables are explicitly forced to **train-median** imputation (only where they were originally missing), to keep them stable and interpretable:
   - `loan_term`
   - `debt_to_income_ratio_ord`

4. **Caching**: The post-imputation feature matrices are cached to disk (`after1.5-train.csv`, `after1.5-test.csv`) so downstream steps can be re-run without repeating KNN.

### 2.45 Log1p Transformation (Model Features)

Several monetary variables in HMDA exhibit strong right-skew. After imputing missing values, we apply a **log1p** transformation to stabilize scale and reduce long-tail influence before outlier capping.

- **Always log1p (fixed set)**: the following non-negative, heavy-tail features are transformed using `log1p(x)`:
  - `loan_amount`
  - `property_value`
  - `total_loan_costs`
  - `origination_charges`
  - `discount_points`
  - `lender_credits`

- **Optional log1p column (data-driven)**: `income` is transformed **only if** the training distribution is sufficiently right-skewed.
  - **Decision rule** (train-only): compute skewness on training values after coercing to numeric, dropping missing, and clipping negatives to 0.
  - If **skewness > 1.0** and there are at least **100** valid training observations, apply log1p to `income`; otherwise, keep it on the original scale.

- **Safety handling for negatives**: because `log1p` requires non-negative inputs, any negative values encountered in these fields are clipped to 0 prior to transformation.

This step is part of the modeling feature pipeline and is applied consistently to train and test, with the optional-column decision learned from the training set only.

### 2.5 Outliers (Winsorization)
To prevent extreme values in variables like `loan_amount` or `income` from distorting model training, we performed **Winsorization** using train-derived bounds:

- **Train-only boundaries**: 1st and 99th percentiles are computed on the imputed training feature matrix, then applied to both train and test using `pandas.Series.clip()`.
- **Continuous-only capping**: Capping is restricted to numeric columns with sufficiently high cardinality (heuristic: `nunique > 10`) to avoid corrupting binary flags and small ordinal codes.
- **Explicit exclusions**: We do **not** cap `debt_to_income_ratio_ord` or `dti_missing_flag`.

### 2.6 One-hot-encoding
We transform the cleaned feature matrix into a modeling-ready design matrix using a `ColumnTransformer`:

- **State binarization**: `state_code` is mapped to a stable binary feature `state_code_bin` (CA $\rightarrow$ 0, TX $\rightarrow$ 1).
- **Numeric vs categorical split (post-cleaning)**: After feature engineering and pruning, we re-evaluate which columns should be treated as numeric vs categorical. Many HMDA fields are numeric-coded categories (1/2/3/...) and are treated as categorical if they have low unique counts.
- **OHE settings**: `OneHotEncoder(handle_unknown='ignore', drop='if_binary')` is used to avoid collinearity for binary features and to safely handle unseen categories.
- **Sparse output**: The final encoded matrices (`X_train_ohe`, `X_test_ohe`) are stored as sparse matrices for memory efficiency.

## 3. Feature Signal Analysis (Target Correlations)

This stage quantifies which features are most associated with approval outcomes while preserving strict train-only evaluation.

All statistics are computed on finalized training artifacts from the pipeline (`X_train_imputed` with `y_train`) to ensure consistency and avoid leakage from test data.

- Train size: **111,992 rows**
- Predictors analyzed: **53** (`40` numeric, `13` categorical)
- Numeric association: **point-biserial correlation** (with Spearman as a robustness check)
- Categorical association: **Cramer's V** and **mutual information**
- Redundancy diagnostic: numeric **Spearman** correlation (`|rho| >= 0.8`)

### 3.1 Key Signal Features

The numeric ranking (see `fig12_numeric_signal_top15.png`) is led by tract/valuation-income structure variables, including `tract_one_to_four_family_homes`, `tract_owner_occupied_units`, `tract_population`, `loan_to_value_ratio`, and `lien_status`, with additional signal from `construction_method` and income-related tract indicators.

Categorical ranking results are summarized in `fig13_categorical_signal_top15.png` and represent the current category-level effects used for interpretation.

These rankings should be used for presentation and modeling decisions.

### 3.2 Multicollinearity / Redundancy Findings

We identified **13 highly correlated numeric pairs** (`|Spearman| >= 0.8`). Most of these relationships occur among co-applicant observation variables, applicant observation variables, and mortgage-structure flags.

This indicates meaningful redundancy in the raw feature space. Practically, the modeling phase can either prune near-duplicate variables for interpretability or retain all encoded information and reduce dimensionality using latent-factor methods.

## 4. Unsupervised Learning (TruncatedSVD)

Given the finalized one-hot encoding pipeline outputs a sparse design matrix, `TruncatedSVD` is the appropriate dimensionality-reduction method for this project.

Unlike dense PCA, SVD operates directly on sparse matrices, preserves computational efficiency, and avoids unnecessary densification risk.

### 4.1 Method and Controls

The unsupervised step is fit on `X_train_ohe` only, with explicit alignment checks against `y_train`. This keeps the full workflow consistent with train-only preprocessing and prevents leakage from test data.

For interpretability, component loadings are mapped back to encoded feature names (`feat_names`, or encoder-derived names when needed).

### 4.2 Results and Practical Use

SVD component selection is based on cumulative explained variance thresholds (90% / 95%) over a practical component range. The resulting latent features (`X_train_svd`, and `X_test_svd` when available) provide a compact representation for downstream modeling.

Top-loading summaries identify which encoded variables drive each latent component, supporting interpretability despite dimensionality reduction.

### 4.3 Modeling Recommendation

Use SVD-reduced features as the primary compact representation and compare them against original feature-selected baselines on validation AUC, F1, PR-AUC, and calibration.

In practice, feature-signal rankings remain the interpretability anchor, while SVD helps reduce redundancy and stabilize model training in the sparse high-dimensional space.